# In what ways do Coding Agents contribute tests? 
#### How frequently do Coding Agents contribute tests? 
#### What types (e.g., unit, integration, end-to-end) are most common?
#### When tests are missing in initial Agentic-PRs, do developers intervene to ensure reliable software testing (via follow-up commits or related PRs)?

In [2]:
import pandas as pd
import ollama
pr_task_type = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_task_type.parquet")


/Users/richardhua/miniconda3/envs/cenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
test_categories = [
  "unit_tests",
  "integration_tests",
  "end_to_end_tests",
  "smoke_tests",
  "performance_tests",
  "regression_tests", 
  "other"
]

SYSTEM_PROMPT = """
You are a highly accurate classifier for software pull requests and commit messages.

Input fields:
- "title": a short PR or commit title
- "reason": an optional explanation of what the PR does

Your task:
Classify the PR into EXACTLY ONE category from this list:

["unit_tests", "integration_tests", "end_to_end_tests", "smoke_tests", "performance_tests", "regression_tests", "none"]

TITLE PRIORITY:
- The TITLE contains the strongest clues.
- If the title clearly references tests, ALWAYS choose a test-related category rather than "none".
- Words that indicate testing activity: "test", "tests", "unit", "coverage", "e2e", "end-to-end", "integration", "cypress", "jest", "pytest", "migrated test", "add tests", "improve tests", "test suite", "test cases".
- If the title starts with "Test", "test:", "tests:", or includes “test”, classify it as testing work unless the reason explicitly contradicts this.

REASON LOGIC:
- Use the reason if it adds helpful detail about the test *type*.
- If the reason is generic, missing, or unclear, rely primarily on the title.
- If the reason mentions fixing a previously known bug, choose "regression_tests".
- If the reason only describes CI, tooling, pre-commit hooks, formatting, or workflows unrelated to tests, choose "none".

CATEGORY DEFINITIONS:

unit_tests
- Tests for individual functions, classes, components, or isolated modules.
- Includes adding missing tests, increasing test coverage, migrating unit test frameworks, or expanding test suites for a single component.

integration_tests
- Tests involving interaction between multiple modules, services, or systems.
- Clues: API + database, service-to-service integration, multi-step internal pipelines.

end_to_end_tests
- Full-system user flows from start to finish.
- Clues: e2e, end-to-end, full workflow, user journey, UI-to-backend, Cypress flows.

smoke_tests
- Minimal/basic tests verifying that the system starts up or that major features work at all.

performance_tests
- Tests for speed, throughput, latency, load, benchmarking, scalability, stress.

regression_tests
- Tests added specifically to prevent a previously known bug, crash, or regression.
- DO NOT choose regression_tests just because the title mentions "error" or "fail".
- Only choose regression_tests if the PR explicitly references:
  - a past bug
  - a crash that occurred before
  - a fix from a previous commit or PR
  - a scenario meant to guard against recurrence

none
- Use ONLY when the PR does not add, modify, or meaningfully change tests.
- Examples:
  - CI config changes
  - pre-commit or workflow updates
  - linting/formatting
  - typo fixes
  - tooling-only updates
- DO NOT choose "none" when the title clearly indicates tests are added or modified.

TIE-BREAKING RULES:
- If title indicates test changes, prefer a test-related category over "none".
- If unsure between unit and integration, choose unit unless multiple systems interact.
- If unsure between integration and end_to_end, choose end_to_end only when the workflow clearly spans a full user-facing or end-to-end path.
- If unsure between unit and regression, choose regression only when previously-known bug or regression is explicitly mentioned.

OUTPUT FORMAT:
- Respond with ONLY the label.
- No punctuation, no quotes, no explanation, no extra words.
"""

def classify_pr(title: str, reason: str, model="qwen2.5-coder"):
    prompt = f"Title: {title}\nReason: {reason}"
    resp = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return resp["message"]["content"].strip()

print(pr_task_type.shape)

for col in pr_task_type.columns:
    print(col)

total_types = {}

for item in pr_task_type.type:
    if item in total_types:
        total_types[item] += 1
    else:
        total_types[item] = 1

print(total_types)

rep = []
pr_test =  pr_task_type[pr_task_type.type == "test"]

for index, row in pr_test.sample(n = 100).iterrows():
    title = row.title
    reason = row.reason
    rep.append((title, reason, classify_pr(title, reason)))

# sample this 25 times, review manually construct a CI

print(rep)


(33596, 6)
agent
id
title
reason
type
confidence
{'fix': 8106, 'feat': 14450, 'chore': 896, 'docs': 3887, 'test': 2356, 'refactor': 2288, 'build': 627, 'ci': 411, 'perf': 340, 'revert': 16, 'style': 188, 'other': 31}
[('Migrate Avatar and AvatarStack tests from Jest to Vitest', 'The PR migrates test files from Jest to Vitest and updates test configurations and test code accordingly, which is primarily related to testing improvements and maintenance rather than adding features or fixing bugs.', 'unit_tests'), ('[alpha_factory] skip fastapi tests when dependency missing', 'The change modifies test behavior to skip certain tests when a dependency is missing, which is related to testing adjustments rather than adding features or fixing bugs.', 'unit_tests'), ('Migrate slot-fill tests from Enzyme to @testing-library/react', 'The PR migrates existing tests from Enzyme to @testing-library/react without adding new features or fixing bugs. It improves test reliability and coverage by refactorin